# Chapter 8. Machine learning for prediction

*Companion notebook to* **Computational Criminology: Research Methods, Ethics, and Reproducible Practice with R and Python** *by Dr Fahad Hameed Khan.*

Run the setup cell once, then run the code cell. Everything uses synthetic data, so nothing here describes a real person. The R version of this chapter is in `code/r/ch08_prediction.R`.

In [ ]:
# Setup: fetch the repository, install what this chapter needs,
# and build the synthetic data. Safe to run more than once.
import os
if not os.path.exists('computational-criminology'):
    !git clone -q https://github.com/drfahadhameedkhan/computational-criminology.git
%cd -q computational-criminology
!pip install -q numpy pandas matplotlib scikit-learn
!python data/synthetic/make_synthetic_data.py

In [ ]:
from pathlib import Path
import pandas as pd

# --- setup: features and target from the synthetic table -------------------
DATA = Path("data/synthetic/ml_table.csv")
df = pd.read_csv(DATA)
features = df[["deprivation", "prior_count", "density"]]
target = df["target"]

# --- book code (Chapter 8) -------------------------------------------------
# Python: a classifier judged on held-out data
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score
X_tr, X_te, y_tr, y_te = train_test_split(features, target,
                          test_size=0.3, random_state=0)   # hold out a test set
clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
p = clf.predict_proba(X_te)[:, 1]                # predicted probabilities
print("AUC      :", round(roc_auc_score(y_te, p), 3))
pred = (p >= 0.5)
print("precision:", round(precision_score(y_te, pred, zero_division=0), 3))
print("recall   :", round(recall_score(y_te, pred, zero_division=0), 3))

# --- exercise hint: leakage on purpose -------------------------------------
# Adding a feature computed over the whole period (including the test rows)
# inflates the AUC. Uncomment to see the effect:
# leaked = (target.groupby(df["group"]).transform("mean"))
# ... refit with 'leaked' included and compare the AUC.

## Exercises

- Coding task. Using two years of incident data aggregated to small areas and months, build a model predicting whether each area exceeds its median burglary count next month. Use a strictly temporal split. Compare your model against the baseline “same as last month”, and report whether the machinery earns its complexity.
- Coding task. Deliberately introduce leakage into the same task, for example by including a feature computed over the full period. Report what happens to the test metrics, then write 200 words on how this leak would look in a real project where no one had introduced it on purpose.
- Coding task. Evaluate your Exercise 1 model using accuracy, AUC, and precision-recall at the decision threshold a police force could actually act on. Explain which measure you would report to a commander and why.
- Your labels are arrests. Design a label audit: name two independent sources of evidence you could use to estimate how far arrest diverges from offending for your offence type, and what pattern in that divergence would make the model unacceptable to deploy.